In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.functions import udf
import pickle  
from pyspark.sql.functions import udf
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
import pandas as pd
from pyspark.sql.functions import pandas_udf
from time import sleep
import numpy as np


sparkConf = SparkConf()
sparkConf.setMaster("spark://spark-master:7077")
sparkConf.setAppName("MLStreamExample")
sparkConf.set("spark.driver.memory", "2g")
sparkConf.set("spark.executor.cores", "1")
sparkConf.set("spark.driver.cores", "1")
# create the Spark session, which is the entry point to the Spark SQL engine.
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()

#load the model 
model_lr = pickle.load(open('/home/jovyan/data/lr_model.pkl', 'rb'))

# create broadcasted model
sc = spark.sparkContext
braodcast_model = sc.broadcast(model_lr)

dataSchema = StructType(
        [StructField("ntp", IntegerType(), True),
         StructField("pgc", IntegerType(), True),
         StructField("dbp", IntegerType(), True),
         StructField("tsft", IntegerType(), True),
         StructField("si", IntegerType(), True),
         StructField("bmi", FloatType(), True),       
         StructField("dpf", FloatType(), True),
         StructField("age", IntegerType(), True),
         StructField("class", StringType(), True)
         ])

df = spark.readStream.schema(dataSchema).format("csv").option("header", "true") \
       .load("/home/jovyan/data/diabetes").drop("class")
df.printSchema()


@udf('integer')
def predict_udf(*cols):
    arr = np.array(braodcast_model.value.predict((cols,)))
    result = np.mean(arr)  # Might be a 0D or 1D array depending on operations
    return int(result.item())  # Explicitly converts to a native Python float

list_of_columns = df.columns
df_prediction = df.withColumn('prediction', predict_udf(*list_of_columns))

query = df_prediction \
    .writeStream \
    .queryName("diabetes_prediction") \
    .format("memory") \
    .outputMode("append") \
    .start()

try:
    for x in range(10):
        spark.sql("SELECT * FROM diabetes_prediction").show()
        sleep(10)
        
except KeyboardInterrupt:
    query.stop()
    # Stop the Spark context
    spark.stop()
    print("Stoped the streaming query and the spark context")

In [ ]:
# Stop the Spark context
spark.stop()